In [49]:
import pandas as pd
import pint
import pint_xarray
import xarray as xr

In [50]:
ureg = pint.UnitRegistry()
ureg.define('terakilometer = 1e12 * kilometer')
ureg.define('milkilometer = 1e6 * kilometer')

pkmGlobal_original = pd.read_csv("image_3_4_data/pkmGlobalTot.csv", delimiter=";", usecols=range(9),skiprows=3).drop(0)
tkmGlobal_original = pd.read_csv("image_3_4_data/tkmGlobalTot.csv", delimiter=";", usecols=range(8),skiprows=3).drop(0)
load_original = pd.read_csv("image_3_4_data/load.csv", delimiter=";", usecols=range(9),skiprows=3).drop(0)
load_original

,Unnamed: 0,Unnamed: 1,walking,cycling,bus,train,car,highspeed train,plane
1,1971,1,1,1,1.284451,1.284451,1.054764,1,1
2,1971,2,1,1,1.245394,1.245394,1.040209,1,1
3,1971,3,1,1,1.528979,1.528979,1.14081,1,1
4,1971,4,1,1,1.681593,1.681593,1.190712,1,1
5,1971,5,1,1,1.628049,1.628049,1.173499,1,1
...,...,...,...,...,...,...,...,...,...
3376,2100,22,1,1,1.021974,1.021974,0.9516566,1,1
3377,2100,23,1,1,0.9477034,0.9477034,0.9198883,1,1
3378,2100,24,1,1,0.909835,0.909835,0.9031619,1,1
3379,2100,25,1,1,1.057553,1.057553,0.9664257,1,1


In [51]:
def convert_to_xarray(df, value_name, variable):
    df = df.rename(columns={"Unnamed: 0":"year","Unnamed: 1":"region"})
    df = (
        df.astype({"year": int, "region": int})  # Convert to integers
        .query("region not in [27, 28] and year in [2019, 2060]")  # Filter regions and years
    )
    # Convert all other columns (modes) to float
    df.iloc[:, 2:] = df.iloc[:, 2:].astype(float)

    # Melt the DataFrame so that transport modes become a single coordinate
    df = df.melt(id_vars=["year", "region"], var_name= variable, value_name=value_name)

    # Convert to xarray DataArray
    dataarray = df.set_index(["region", "year", "mode"]).to_xarray()[value_name]
    return dataarray

In [52]:
pkmGlobal = convert_to_xarray(pkmGlobal_original, value_name = "pkm", variable = "mode")
tkmGlobal = convert_to_xarray(tkmGlobal_original, value_name = "tkm", variable = "mode")
load = convert_to_xarray(load_original, value_name = "load", variable = "mode")


In [53]:
# Convert units
pkmGlobal = pkmGlobal.pint.quantify('terakilometer') 
pkmGlobal= pkmGlobal.pint.to("km")